# Phase 6: External Validation on ST003514 (NIST SRM 1950)

**Version: 1.0** | 2026-03-08

**Question:** Does our elution sequence model generalize to an independent dataset from a different instrument/column?

| | Training Data | External Validation |
|---|---|---|
| **Source** | 4 clinical cohorts | NIST SRM 1950 (ST003514) |
| **Instrument** | SCIEX TripleTOF 6600+ | Agilent 6545 QTOF |
| **Column** | Waters CSH C18 | Different C18 |
| **Alignment** | MS-DIAL (our settings) | MS-DIAL (their settings) |

**Prerequisites:** Run `01_train_models.ipynb` first — this notebook loads saved model checkpoints.

**Expected outcome:** Performance drop (different instrument/column/alignment), but if the model learned generalizable elution chemistry, it should still outperform random (1.1% top-1).

**Changelog:**
- v1.0: Split from `02_post_training_experiments.ipynb` into standalone notebook

## 1. Setup

In [ ]:
import subprocess, sys, os

import torch
import torch.nn as nn
print(f"PyTorch {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

try:
    import pyarrow
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pyarrow"])

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import json

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# === Paths ===
BASE_DIR = "/content/drive/MyDrive/0000 Fun with coding/088 Lights-Out R01 Grant/Specific Aim 1/poc3_elution_sequence"
EXPERIMENT_DIR = f"{BASE_DIR}/02_external_validation"
OUTPUT_DIR = f"{EXPERIMENT_DIR}/outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Model checkpoints (from training notebook — in 01_train_models subfolder)
TRAIN_DIR = f"{BASE_DIR}/01_train_models"
LSTM_CHECKPOINT = f"{TRAIN_DIR}/outputs/lstm_best.pt"
TRANSFORMER_CHECKPOINT = f"{TRAIN_DIR}/outputs/transformer_best.pt"

# Data
TOKENIZED_PATH = f"{TRAIN_DIR}/tokenized_features.parquet"  # training data (for comparison)
ST003514_PATH = f"{EXPERIMENT_DIR}/st003514_long.parquet"
ST003514_META_PATH = f"{EXPERIMENT_DIR}/st003514_metabolites.csv"

print(f"Experiment dir: {EXPERIMENT_DIR}")
print(f"LSTM checkpoint: {LSTM_CHECKPOINT}")
print(f"ST003514 data: {ST003514_PATH}")

In [ ]:
# === Configuration (must match training notebook) ===
RANDOM_SEED = 42
CONTEXT_LENGTH = 64
EMBEDDING_DIM = 64
HIDDEN_DIM = 128
NUM_LAYERS = 2
DROPOUT = 0.1
NUM_HEADS = 4
FF_DIM = 256
MZ_BIN_WIDTH = 10  # Da
MASS_DEFECT_BINS = 20
RT_BIN_WIDTH = 3   # seconds

RT_GAP_BINS = [-0.001, 0.1, 0.5, 1.0, 2.0, 5.0, 15.0, 9999]
RT_GAP_LABELS = ["co-elute", "0.1-0.5s", "0.5-1s", "1-2s", "2-5s", "5-15s", ">15s"]
INTENSITY_RANK_BINS = [-0.001, 0.01, 0.05, 0.20, 0.50, 1.001]
INTENSITY_RANK_LABELS = ["top1%", "top5%", "top20%", "top50%", "low"]
POLARITY_MAP = {"pos": 0, "neg": 1, "unk": 2}
RT_GAP_MAP = {label: i for i, label in enumerate(RT_GAP_LABELS)}
INTENSITY_MAP = {label: i for i, label in enumerate(INTENSITY_RANK_LABELS)}

np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

## 2. Model Definitions & Checkpoint Loading

In [ ]:
class MultiFieldEmbedding(nn.Module):
    def __init__(self, embedding_dim, max_mz_bin=120, max_md_bin=20,
                 max_rt_gap=7, max_polarity=3, max_intensity=5):
        super().__init__()
        self.mz_embed = nn.Embedding(max_mz_bin, embedding_dim)
        self.md_embed = nn.Embedding(max_md_bin, embedding_dim)
        self.gap_embed = nn.Embedding(max_rt_gap, embedding_dim)
        self.pol_embed = nn.Embedding(max_polarity, embedding_dim)
        self.int_embed = nn.Embedding(max_intensity, embedding_dim)

    def forward(self, mz, md, gap, pol, intensity):
        return (self.mz_embed(mz) + self.md_embed(md) + self.gap_embed(gap)
                + self.pol_embed(pol) + self.int_embed(intensity))


class LSTMModel(nn.Module):
    def __init__(self, num_mz_classes, embedding_dim=64, hidden_dim=128,
                 num_layers=2, dropout=0.1, **embed_kw):
        super().__init__()
        self.embedding = MultiFieldEmbedding(embedding_dim, **embed_kw)
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, num_layers,
                           dropout=dropout if num_layers > 1 else 0, batch_first=True)
        self.dropout = nn.Dropout(dropout)
        self.head = nn.Linear(hidden_dim, num_mz_classes)

    def forward(self, mz, md, gap, pol, intensity, hidden=None):
        x = self.embedding(mz, md, gap, pol, intensity)
        out, hidden = self.lstm(x, hidden)
        return self.head(self.dropout(out[:, -1, :])), hidden


class TransformerModel(nn.Module):
    def __init__(self, num_mz_classes, embedding_dim=64, num_heads=4,
                 ff_dim=256, num_layers=2, dropout=0.1,
                 context_length=64, **embed_kw):
        super().__init__()
        self.embedding = MultiFieldEmbedding(embedding_dim, **embed_kw)
        self.pos_embed = nn.Embedding(context_length, embedding_dim)
        self.dropout = nn.Dropout(dropout)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embedding_dim, nhead=num_heads, dim_feedforward=ff_dim,
            dropout=dropout, batch_first=True, norm_first=True)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.head = nn.Linear(embedding_dim, num_mz_classes)
        self.register_buffer("causal_mask",
            torch.triu(torch.ones(context_length, context_length), diagonal=1).bool())

    def forward(self, mz, md, gap, pol, intensity):
        B, T = mz.shape
        x = self.embedding(mz, md, gap, pol, intensity)
        x = x + self.pos_embed(torch.arange(T, device=mz.device).unsqueeze(0))
        x = self.dropout(x)
        x = self.transformer(x, mask=self.causal_mask[:T, :T])
        return self.head(x[:, -1, :])

In [ ]:
# === Load trained model checkpoints ===
lstm_ckpt = torch.load(LSTM_CHECKPOINT, map_location=device, weights_only=False)
MAX_MZ_BIN = lstm_ckpt["config"]["num_classes"]
print(f"Max m/z bin: {MAX_MZ_BIN}")

embed_kw = dict(max_mz_bin=MAX_MZ_BIN, max_md_bin=20, max_rt_gap=7, max_polarity=3, max_intensity=5)

lstm_model = LSTMModel(MAX_MZ_BIN, EMBEDDING_DIM, HIDDEN_DIM, NUM_LAYERS, DROPOUT, **embed_kw)
lstm_model.load_state_dict(lstm_ckpt["model_state_dict"])
lstm_model = lstm_model.to(device)
lstm_model.eval()
print(f"LSTM loaded \u2014 test metrics: {lstm_ckpt['test_metrics']}")

tfm_ckpt = torch.load(TRANSFORMER_CHECKPOINT, map_location=device, weights_only=False)
tfm_model = TransformerModel(MAX_MZ_BIN, EMBEDDING_DIM, NUM_HEADS, FF_DIM, NUM_LAYERS, DROPOUT,
                             CONTEXT_LENGTH, **embed_kw)
tfm_model.load_state_dict(tfm_ckpt["model_state_dict"])
tfm_model = tfm_model.to(device)
tfm_model.eval()
print(f"Transformer loaded \u2014 test metrics: {tfm_ckpt['test_metrics']}")

## 3. Tokenize ST003514

In [ ]:
# === Load and tokenize ST003514 ===
st = pd.read_parquet(ST003514_PATH)
print(f"ST003514 raw: {len(st):,} rows, {st.sample_id.nunique()} samples, {st.feature_id.nunique()} features")

# Filter to detected features with valid RT/m/z
st = st[(st.intensity > 0) & st.mz.notna() & st.rt.notna()].copy()
print(f"Detected features: {len(st):,} rows")

# === Tokenize using same scheme as training data ===
# m/z bin
st["mz_bin"] = (st["mz"] // MZ_BIN_WIDTH).astype(int)

# Mass defect bin
mass_defect = st["mz"] - np.floor(st["mz"])
st["md_bin"] = (mass_defect * MASS_DEFECT_BINS).astype(int).clip(upper=MASS_DEFECT_BINS - 1)

# Polarity
st["polarity_tok"] = st["polarity"].map({"(+) ESI": "pos", "(-) ESI": "neg"}).fillna("unk")
st["polarity_idx"] = st["polarity_tok"].map(POLARITY_MAP).fillna(2).astype(int)

# Sort by RT within each sample, compute RT gaps
st = st.sort_values(["sample_id", "rt", "mz"])
st["rt_gap"] = st.groupby("sample_id")["rt"].diff() * 60  # min -> seconds
st["rt_gap"] = st["rt_gap"].fillna(0).clip(lower=0)
st["rt_gap_bin"] = pd.cut(st["rt_gap"], bins=RT_GAP_BINS, labels=RT_GAP_LABELS,
                           right=False, include_lowest=True)
st["rt_gap_idx"] = st["rt_gap_bin"].astype(str).map(RT_GAP_MAP).fillna(0).astype(int)

# Intensity rank within each sample
st["intensity_rank"] = st.groupby("sample_id")["intensity"].transform(
    lambda x: pd.cut(1 - x.rank(pct=True), bins=INTENSITY_RANK_BINS,
                     labels=INTENSITY_RANK_LABELS, right=True, include_lowest=True)
)
st["intensity_idx"] = st["intensity_rank"].astype(str).map(INTENSITY_MAP).fillna(4).astype(int)

# RT bin (prediction target)
st["rt_bin"] = (st["rt"] * 60 / RT_BIN_WIDTH).astype(int)

# Sequence position
st["seq_pos"] = st.groupby("sample_id").cumcount()
st["study"] = "st003514"
st["sample_type"] = "analytical"  # all NIST SRM 1950 samples

# Check m/z bin range vs training vocab
st_max_mz = st["mz_bin"].max()
print(f"\nST003514 m/z bin range: {st['mz_bin'].min()}-{st_max_mz}")
print(f"Training vocab max m/z bin: {MAX_MZ_BIN - 1}")
oov = (st["mz_bin"] >= MAX_MZ_BIN).sum()
print(f"Out-of-vocabulary m/z bins: {oov}/{len(st)} ({oov/len(st)*100:.1f}%)")

# Clip OOV bins to max training bin
st["mz_bin"] = st["mz_bin"].clip(upper=MAX_MZ_BIN - 1)

print(f"\nTokenized ST003514: {len(st):,} rows")
print(f"  Samples: {st.sample_id.nunique()}")
print(f"  Features per sample: {st.groupby('sample_id').size().describe()[['mean','min','max']].to_dict()}")

## 4. Evaluate Models on ST003514

In [ ]:
def build_sample_arrays(df):
    """Convert DataFrame to list of per-sample numpy arrays."""
    samples = []
    for (study, sid), group in df.groupby(["study", "sample_id"]):
        g = group.sort_values("seq_pos" if "seq_pos" in group.columns else "rt")
        samples.append({
            "study": study, "sample_id": sid,
            "sample_type": g["sample_type"].iloc[0] if "sample_type" in g.columns else "analytical",
            "mz_bin": g["mz_bin"].values.astype(np.int64),
            "md_bin": g["md_bin"].values.astype(np.int64),
            "rt_gap_idx": g["rt_gap_idx"].values.astype(np.int64),
            "polarity_idx": g["polarity_idx"].values.astype(np.int64),
            "intensity_idx": g["intensity_idx"].values.astype(np.int64),
        })
    return samples


def evaluate_on_samples(model, sample_arrays, context_length=CONTEXT_LENGTH, top_k=(1, 3, 5, 10)):
    """Evaluate model on sample arrays using sliding window."""
    model.eval()
    all_results = []
    
    with torch.no_grad():
        for s in sample_arrays:
            n = len(s["mz_bin"])
            if n <= context_length:
                continue
            
            for pos in range(context_length, n):
                start = pos - context_length
                mz = torch.tensor(s["mz_bin"][start:pos], dtype=torch.long).unsqueeze(0).to(device)
                md = torch.tensor(s["md_bin"][start:pos], dtype=torch.long).unsqueeze(0).to(device)
                gap = torch.tensor(s["rt_gap_idx"][start:pos], dtype=torch.long).unsqueeze(0).to(device)
                pol = torch.tensor(s["polarity_idx"][start:pos], dtype=torch.long).unsqueeze(0).to(device)
                inten = torch.tensor(s["intensity_idx"][start:pos], dtype=torch.long).unsqueeze(0).to(device)
                target = int(s["mz_bin"][pos])
                
                if isinstance(model, LSTMModel):
                    logits, _ = model(mz, md, gap, pol, inten)
                else:
                    logits = model(mz, md, gap, pol, inten)
                
                pred = logits.argmax(dim=1).item()
                topk_preds = {}
                for k in top_k:
                    _, tk = logits.topk(k, dim=1)
                    topk_preds[f"top{k}"] = target in tk[0].cpu().numpy()
                
                all_results.append({
                    "study": s["study"], "sample_id": s["sample_id"],
                    "sample_type": s.get("sample_type", "analytical"),
                    "position": pos, "target": target, "pred": pred,
                    "mae_da": abs(pred - target) * MZ_BIN_WIDTH,
                    **topk_preds,
                })
    
    return pd.DataFrame(all_results)

In [ ]:
# === Evaluate LSTM and Transformer on ST003514 ===
st_arrays = build_sample_arrays(st)
print(f"Built {len(st_arrays)} sample arrays for ST003514")

print("\n--- LSTM on ST003514 ---")
lstm_st_results = evaluate_on_samples(lstm_model, st_arrays)
n = len(lstm_st_results)
if n > 0:
    print(f"  n={n:,}  top1={lstm_st_results['top1'].mean():.4f}  top5={lstm_st_results['top5'].mean():.4f}  MAE={lstm_st_results['mae_da'].mean():.0f}Da")

print("\n--- Transformer on ST003514 ---")
tfm_st_results = evaluate_on_samples(tfm_model, st_arrays)
n = len(tfm_st_results)
if n > 0:
    print(f"  n={n:,}  top1={tfm_st_results['top1'].mean():.4f}  top5={tfm_st_results['top5'].mean():.4f}  MAE={tfm_st_results['mae_da'].mean():.0f}Da")

## 5. Compare with Training-Set Performance

In [ ]:
lstm_st_metrics = {"top1": lstm_st_results["top1"].mean(), "top3": lstm_st_results["top3"].mean(),
                   "top5": lstm_st_results["top5"].mean(), "top10": lstm_st_results["top10"].mean(),
                   "mae_da": lstm_st_results["mae_da"].mean(), "n": len(lstm_st_results)}
tfm_st_metrics = {"top1": tfm_st_results["top1"].mean(), "top3": tfm_st_results["top3"].mean(),
                  "top5": tfm_st_results["top5"].mean(), "top10": tfm_st_results["top10"].mean(),
                  "mae_da": tfm_st_results["mae_da"].mean(), "n": len(tfm_st_results)}

print("\n" + "=" * 75)
print(f"{'Dataset':<25} {'Model':<15} {'Top-1':<8} {'Top-5':<8} {'MAE(Da)':<8}")
print("-" * 75)
print(f"{'Training cohorts (test)':<25} {'LSTM':<15} {lstm_ckpt['test_metrics']['top1']:.4f}  {lstm_ckpt['test_metrics']['top5']:.4f}  {lstm_ckpt['test_metrics']['mae_da']:.0f}")
print(f"{'Training cohorts (test)':<25} {'Transformer':<15} {tfm_ckpt['test_metrics']['top1']:.4f}  {tfm_ckpt['test_metrics']['top5']:.4f}  {tfm_ckpt['test_metrics']['mae_da']:.0f}")
print(f"{'ST003514 (external)':<25} {'LSTM':<15} {lstm_st_metrics['top1']:.4f}  {lstm_st_metrics['top5']:.4f}  {lstm_st_metrics['mae_da']:.0f}")
print(f"{'ST003514 (external)':<25} {'Transformer':<15} {tfm_st_metrics['top1']:.4f}  {tfm_st_metrics['top5']:.4f}  {tfm_st_metrics['mae_da']:.0f}")
print(f"{'Random baseline':<25} {'':<15} 0.0109  \u2014       1965")
print("=" * 75)

# Generalization ratio
for name, train_m, ext_m in [("LSTM", lstm_ckpt['test_metrics'], lstm_st_metrics),
                              ("Transformer", tfm_ckpt['test_metrics'], tfm_st_metrics)]:
    ratio = ext_m['top1'] / train_m['top1'] if train_m['top1'] > 0 else 0
    print(f"\n{name} generalization: {ratio:.1%} of training-set performance retained")

## 6. Figure: External Validation

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Panel A: Top-k accuracy comparison (bar chart)
ax = axes[0]
x = np.arange(4)  # top-1, top-3, top-5, top-10
width = 0.2
labels = ["Top-1", "Top-3", "Top-5", "Top-10"]
keys = ["top1", "top3", "top5", "top10"]

lstm_train = [lstm_ckpt['test_metrics'].get(k, 0) for k in keys]
tfm_train = [tfm_ckpt['test_metrics'].get(k, 0) for k in keys]
lstm_ext = [lstm_st_metrics.get(k, 0) for k in keys]
tfm_ext = [tfm_st_metrics.get(k, 0) for k in keys]

ax.bar(x - 1.5*width, lstm_train, width, label="LSTM (train cohorts)", color="tab:blue", alpha=0.8)
ax.bar(x - 0.5*width, lstm_ext, width, label="LSTM (ST003514)", color="tab:blue", alpha=0.4, hatch="//")
ax.bar(x + 0.5*width, tfm_train, width, label="Transformer (train cohorts)", color="tab:orange", alpha=0.8)
ax.bar(x + 1.5*width, tfm_ext, width, label="Transformer (ST003514)", color="tab:orange", alpha=0.4, hatch="//")

ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_ylabel("Accuracy")
ax.set_title("A. Prediction Accuracy: Training vs External")
ax.legend(fontsize=8)
ax.grid(axis="y", alpha=0.3)

# Panel B: Per-position accuracy (does model improve with more context?)
ax = axes[1]
for name, results, color in [("LSTM", lstm_st_results, "tab:blue"),
                              ("Transformer", tfm_st_results, "tab:orange")]:
    if len(results) == 0:
        continue
    results = results.copy()
    results["pos_bin"] = (results["position"] // 20) * 20
    pos_acc = results.groupby("pos_bin")["top1"].mean()
    ax.plot(pos_acc.index, pos_acc.values, label=name, color=color, alpha=0.8)

ax.set_xlabel("Position in elution sequence")
ax.set_ylabel("Top-1 accuracy")
ax.set_title("B. Accuracy vs Position (ST003514)")
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/phase6_external_validation.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {OUTPUT_DIR}/phase6_external_validation.png")

## 7. Save Results

In [ ]:
results = {
    "experiment": "Phase 6: External Validation",
    "external_dataset": "ST003514 (NIST SRM 1950)",
    "external_instrument": "Agilent 6545 QTOF",
    "training_instrument": "SCIEX TripleTOF 6600+",
    "lstm": {
        "training_test": lstm_ckpt["test_metrics"],
        "external": lstm_st_metrics,
        "generalization_ratio": lstm_st_metrics["top1"] / lstm_ckpt["test_metrics"]["top1"] if lstm_ckpt["test_metrics"]["top1"] > 0 else 0,
    },
    "transformer": {
        "training_test": tfm_ckpt["test_metrics"],
        "external": tfm_st_metrics,
        "generalization_ratio": tfm_st_metrics["top1"] / tfm_ckpt["test_metrics"]["top1"] if tfm_ckpt["test_metrics"]["top1"] > 0 else 0,
    },
}

with open(f"{OUTPUT_DIR}/phase6_results.json", "w") as f:
    json.dump(results, f, indent=2, default=str)

lstm_st_results.to_parquet(f"{OUTPUT_DIR}/phase6_lstm_detailed.parquet", index=False)
tfm_st_results.to_parquet(f"{OUTPUT_DIR}/phase6_tfm_detailed.parquet", index=False)

print("All results saved.")
print(f"Output directory: {OUTPUT_DIR}")
for f_name in os.listdir(OUTPUT_DIR):
    size = os.path.getsize(f"{OUTPUT_DIR}/{f_name}")
    print(f"  {f_name:<45} {size/1024:.0f} KB")